# SI7011 · Sesión 04 · Entrenar los *checkpoints* de SimCLR (solo profesor)

Produce los dos archivos del dataset de Kaggle [`juanmartinezv4399/si7011-s04-simclr`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s04-simclr), que cargan los notebooks 4.4 y 4.5:

| Archivo | Qué es |
|---|---|
| `simclr_resnet18_cifar10.pt` | SimCLR con las vistas completas del notebook 4.4 |
| `simclr_sin_color.pt` | lo mismo, sin *color jitter* ni escala de grises (sección 6 del 4.4) |

Las vistas, el modelo y la pérdida son **copias exactas** de las celdas del notebook 4.4.

**Cómo correrlo en Kaggle**
1. *Add Input*: `pankrzysiu/cifar10-python`. *Settings*: GPU (T4 o P100).
2. **Save Version → Save & Run All (Commit)**: corre en segundo plano y no depende de que el navegador siga abierto.
3. Al terminar, pestaña *Output* de esa versión → **New Dataset** → nombre `si7011-s04-simclr`, **público**.

**Si no alcanza el tiempo.** El notebook guarda un archivo `*_parcial.pt` al final de cada época y se detiene solo antes de `MAX_HORAS`. Para seguir: en una versión nueva, *Add Input* → *Your Work* → la salida de la versión anterior, y otra vez *Save & Run All*. Retoma en la época donde quedó, y copia a la salida los archivos que ya estaban terminados, de modo que la última versión siempre tiene los dos.

Después de la primera época imprime cuánto va a tardar todo. Si es demasiado, baja `EPOCAS` (con 30 épocas, el $k$-NN ya queda muy por encima del encoder aleatorio).

In [ ]:
import math, os, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets
from torchvision.models import resnet18

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = device.type == 'cuda'
torch.backends.cudnn.benchmark = True                    # tamaños fijos: cuDNN elige la convolución más rápida
print('PyTorch', torch.__version__, '· device', device,
      '·', torch.cuda.get_device_name() if amp else 'sin GPU: esto va a tardar muchísimo')

## Configuración

In [ ]:
EPOCAS = 30                  # por corrida; el notebook 4.4 lee este número del checkpoint
N, TAU = 512, 0.2            # los mismos del notebook 4.4: 2N = 1024 vistas por lote
MAX_HORAS = 11.0             # Kaggle corta a las 12 h: guarda y se detiene antes
CORRIDAS = [('simclr_resnet18_cifar10.pt', True),       # (archivo, con color jitter)
            ('simclr_sin_color.pt', False)]
INICIO = time.time()

## Datos (cargador del notebook 4.4)

In [ ]:
import os, pickle, tarfile
from pathlib import Path

KAGGLE_HANDLE = 'pankrzysiu/cifar10-python'          # CIFAR-10 en el formato original para Python
ROOTS = [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def find_batches(roots):
    for root in [r for r in roots if r.exists()]:
        hit = next(root.rglob('data_batch_1'), None)
        if hit is None:                                    # algunas copias traen solo el comprimido
            tgz = next(root.rglob('cifar-10-python.tar.gz'), None)
            if tgz is not None:
                print('Descomprimiendo', tgz, '→ data/')
                with tarfile.open(tgz) as t:
                    t.extractall('data')
                hit = next(Path('data').rglob('data_batch_1'), None)
        if hit is not None:
            return hit.parent
    return None

def locate_cifar10():
    folder = find_batches(ROOTS)
    if folder is not None:
        return folder, 'input de Kaggle' if str(folder).startswith('/kaggle/input') else 'archivos locales'
    if kaggle_token() or Path('/kaggle/input').exists():
        try:
            import kagglehub                               # preinstalado en Colab y Kaggle
            folder = find_batches([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
            if folder is not None:
                return folder, 'kagglehub'
        except Exception as e:
            print('kagglehub no disponible:', type(e).__name__, '→ se descarga del sitio original')
    datasets.CIFAR10('data', download=True)                # el mismo formato, en data/cifar-10-batches-py
    return find_batches([Path('data')]), 'torchvision (sitio original)'

def read_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    # cada fila: 3 072 bytes = R (1 024) | G (1 024) | B (1 024) → 3 × 32 × 32 → H × W × C, como un PNG
    x = np.asarray(d[b'data'], dtype=np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return x, np.asarray(d[b'labels'], dtype=np.int64)

def load_cifar10():
    folder, source = locate_cifar10()
    parts = [read_batch(folder / f'data_batch_{i}') for i in range(1, 6)]   # 5 × 10 000 de entrenamiento
    x_test, y_test = read_batch(folder / 'test_batch')                      # 10 000 de prueba
    with open(folder / 'batches.meta', 'rb') as f:
        classes = [c.decode() for c in pickle.load(f, encoding='bytes')[b'label_names']]
    print('Fuente:', source, '·', folder)
    return np.concatenate([x for x, _ in parts]), np.concatenate([y for _, y in parts]), x_test, y_test, classes

x_train_raw, y_train_raw, x_test_raw, y_test_raw, CLASSES = load_cifar10()
x_train_raw.shape, x_test_raw.shape

In [ ]:
X_train = torch.from_numpy(x_train_raw).permute(0, 3, 1, 2).to(device)    # uint8, N × 3 × 32 × 32
X_test = torch.from_numpy(x_test_raw).permute(0, 3, 1, 2).to(device)
y_train, y_test = torch.from_numpy(y_train_raw), torch.from_numpy(y_test_raw)   # solo para medir
MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)
norm = lambda x: (x - MEAN) / STD                                          # x ∈ [0, 1] → normalizado
X_train.shape, X_test.shape

## Vistas, modelo y pérdida (copiados del notebook 4.4)

In [ ]:
def recorte_espejo(x, escala=(0.2, 1.0), razon=(3 / 4, 4 / 3)):
    # recorte aleatorio reescalado + espejo, con una transformación afín por imagen (grid_sample)
    B = len(x)
    area = torch.empty(B, device=x.device).uniform_(*escala)
    r = torch.exp(torch.empty(B, device=x.device).uniform_(math.log(razon[0]), math.log(razon[1])))
    w, h = (area * r).sqrt().clamp(max=1), (area / r).sqrt().clamp(max=1)     # fracción del ancho y del alto
    cx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - w)                    # centro, en coordenadas [−1, 1]
    cy = (torch.rand(B, device=x.device) * 2 - 1) * (1 - h)
    espejo = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = w * espejo, cx, h, cy
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)

def gris(x):
    return (0.299 * x[:, :1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:]).expand_as(x)

def color_jitter(x, b=0.4, c=0.4, s=0.4, hue=0.1, p=0.8):
    B = len(x); u = lambda a: torch.empty(B, 1, 1, 1, device=x.device).uniform_(1 - a, 1 + a)
    y = x * u(b)                                                                   # brillo
    m = gris(y).mean((1, 2, 3), keepdim=True); y = (y - m) * u(c) + m              # contraste
    gy = gris(y); y = (y - gy) * u(s) + gy                                         # saturación
    # tono: rotación en el plano de color (I, Q) del espacio YIQ
    yiq = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=x.device)
    ang = torch.empty(B, device=x.device).uniform_(-hue, hue) * 2 * math.pi
    rot = torch.zeros(B, 3, 3, device=x.device); rot[:, 0, 0] = 1
    rot[:, 1, 1], rot[:, 1, 2], rot[:, 2, 1], rot[:, 2, 2] = ang.cos(), -ang.sin(), ang.sin(), ang.cos()
    M = torch.linalg.inv(yiq) @ rot @ yiq                                          # B × 3 × 3
    y = torch.einsum('bij,bjhw->bihw', M, y).clamp(0, 1)
    aplicar = (torch.rand(B, 1, 1, 1, device=x.device) < p)
    return torch.where(aplicar, y, x)

def vista(xb, color=True):
    # x̃ = t(x), t ∼ 𝒯: una vista aleatoria por imagen; uint8 → normalizada
    x = recorte_espejo(xb.float() / 255)
    if color:
        x = color_jitter(x)
        x = torch.where(torch.rand(len(x), 1, 1, 1, device=x.device) < 0.2, gris(x), x)
    return norm(x.clamp(0, 1))

In [ ]:
def resnet18_cifar():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)   # 32×32 no se reduce al entrar
    m.maxpool = nn.Identity()
    m.fc = nn.Identity()                                              # g(x) = h ∈ ℝ^512
    return m

def cabeza():
    return nn.Sequential(nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Linear(512, 128))   # u = p(h) ∈ ℝ^128

def info_nce(u, tau):
    u = F.normalize(u, dim=1)                          # u / ‖u‖: el producto punto queda como coseno
    S = u @ u.T / tau                                  # s_ij para las 2N × 2N parejas
    S = S.masked_fill(torch.eye(len(u), dtype=torch.bool, device=u.device), float('-inf'))   # k ≠ i
    pos = torch.arange(len(u), device=u.device).roll(len(u) // 2)                           # j(i): la otra vista
    loss = F.cross_entropy(S, pos)                     # promedio de L_i = −log softmax(S_i)_{j(i)}
    acierto = (S.argmax(1) == pos).float().mean()      # ¿el más parecido es la otra vista?
    return loss, acierto

## Entrenar, guardando cada época
`buscar` mira la carpeta de trabajo y los *inputs*. Un archivo final que ya existe no se vuelve a entrenar; un `*_parcial.pt` se retoma con el optimizador, el *scheduler* y la historia que tenía.

In [ ]:
ROOTS = [Path('.'), Path('/kaggle/input'), Path('data')]

def buscar(nombre):
    for d in ROOTS:
        hits = sorted(d.glob(f'**/{nombre}')) if d.exists() else []
        if hits:
            return hits[0]

def a_la_salida(ruta):
    # la salida de esta versión debe tener los dos archivos, aunque uno venga de una versión anterior
    if ruta.resolve().parent != Path('.').resolve():
        shutil.copy(ruta, ruta.name)

def preparar(color):
    torch.manual_seed(0)
    g, p = resnet18_cifar().to(device), cabeza().to(device)
    opt = torch.optim.SGD([*g.parameters(), *p.parameters()], lr=0.06 * N / 256, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    pasos_ep = len(X_train) // N
    total, warm = EPOCAS * pasos_ep, max(1, EPOCAS * pasos_ep // 10)        # el mismo schedule del 4.4
    f = lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, f)
    return g, p, opt, sched, torch.amp.GradScaler(enabled=amp)

def entrenar_corrida(nombre, color):
    hecho = buscar(nombre)
    if hecho is not None:
        print(f'{nombre}: ya existe ({hecho}), no se entrena'); a_la_salida(hecho); return True
    g, p, opt, sched, scaler = preparar(color)
    parcial_nombre = nombre.replace('.pt', '_parcial.pt')
    parcial, hist, inicio = buscar(parcial_nombre), [], 0
    if parcial is not None:
        ck = torch.load(parcial, map_location=device)
        g.load_state_dict(ck['g']); p.load_state_dict(ck['p']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        hist, inicio = ck['historia'], ck['epoca']
        print(f'{nombre}: retomo desde {parcial}, época {inicio}')
    pasos_ep = len(X_train) // N
    g.train(); p.train()
    for ep in range(inicio, EPOCAS):
        t0, perdidas, aciertos = time.time(), [], []
        for b in torch.randperm(len(X_train), device=device)[:pasos_ep * N].split(N):
            xb = X_train[b]
            vistas = torch.cat([vista(xb, color), vista(xb, color)])
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                loss, acierto = info_nce(p(g(vistas)), TAU)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perdidas.append(loss.item()); aciertos.append(acierto.item())
        hist.append({'perdida': float(np.mean(perdidas)), 'acierto_positivo': float(np.mean(aciertos))})
        torch.save({'g': g.state_dict(), 'p': p.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'scaler': scaler.state_dict(), 'historia': hist, 'epoca': ep + 1}, parcial_nombre)
        dt = time.time() - t0
        horas = (time.time() - INICIO) / 3600
        print(f"{nombre} · época {ep + 1}/{EPOCAS} · pérdida {hist[-1]['perdida']:.3f} · "
              f"acierto del positivo {hist[-1]['acierto_positivo']:.3f} · {dt:.0f} s · "
              f"faltan ~{(EPOCAS - ep - 1) * dt / 60:.0f} min de esta corrida · {horas:.1f} h en esta sesión")
        if ep == inicio:
            pendientes = sum(EPOCAS for n, _ in CORRIDAS if buscar(n) is None) - inicio
            print(f'   estimado: ~{pendientes * dt / 3600:.1f} h para todo lo que falta (límite {MAX_HORAS} h por sesión)')
        if horas + dt / 3600 > MAX_HORAS and ep + 1 < EPOCAS:
            print(f'   me detengo para no pasar de {MAX_HORAS} h. Para seguir, agrega esta salida como input y vuelve a correr.')
            return False
    torch.save({'g': g.state_dict(), 'p': p.state_dict(), 'historia': hist, 'tau': TAU, 'N': N,
                'epocas': EPOCAS, 'color': color}, nombre)              # el formato que cargan 4.4 y 4.5
    Path(parcial_nombre).unlink(missing_ok=True)
    print(f'{nombre}: listo')
    return True

for nombre, color in CORRIDAS:
    if not entrenar_corrida(nombre, color):
        break

## Comprobar los archivos
El mismo $k$-NN del notebook 4.4 ($k = 20$, coseno, voto ponderado), comparado con un encoder aleatorio. Las etiquetas solo se usan aquí.

In [ ]:
@torch.no_grad()
def embeddings(g, X, chunk=1000):
    g.eval()
    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
        return torch.cat([g(norm(X[i:i + chunk].float() / 255)).float() for i in range(0, len(X), chunk)])

@torch.no_grad()
def knn(g, k=20, t=0.07):
    Htr, Hte = F.normalize(embeddings(g, X_train), dim=1), F.normalize(embeddings(g, X_test), dim=1)
    ytr = y_train.to(device); pred = []
    for i in range(0, len(Hte), 1000):
        sim, j = (Hte[i:i + 1000] @ Htr.T).topk(k, dim=1)
        pred.append(torch.zeros(len(j), 10, device=device).scatter_add_(1, ytr[j], (sim / t).exp()).argmax(1))
    return (torch.cat(pred).cpu() == y_test).float().mean().item()

torch.manual_seed(0)
print(f'aleatorio · k-NN {knn(resnet18_cifar().to(device)):.3f}')
for nombre, _ in CORRIDAS:
    if Path(nombre).exists():
        ck = torch.load(nombre, map_location=device)
        g = resnet18_cifar().to(device); g.load_state_dict(ck['g'])
        h = ck['historia'][-1]
        print(f"{nombre} · {ck['epocas']} épocas · pérdida {h['perdida']:.3f} · acierto del positivo "
              f"{h['acierto_positivo']:.3f} · k-NN {knn(g):.3f} · {os.path.getsize(nombre) / 1e6:.0f} MB")
    else:
        print(f'{nombre}: todavía no está terminado')